# Exploratory Colab experiment

> Cleaned archive of the original graduation-project notebook. For new leakage-aware runs, use the reusable pipeline under src/ and scripts/.


In [ ]:
# 1) Drive bağlantısı ve kütüphaneler

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight

# 2) Sabitler
IMG_SIZE   = 299
BATCH_SIZE = 32
EPOCHS     = 200
START_EPOCH = 150  # Eğitime 150’den devam edeceğiz

BASE_DIR  = 'data/split'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_SPLIT = 0.15

# 3) Data Augmentation (yumuşatılmış)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VAL_SPLIT,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,
    fill_mode='nearest'
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# 4) Güncellenmiş class_weight hesapla
y_train = train_gen.classes
class_weights_array = compute_class_weight(class_weight='balanced',
                                           classes=np.unique(y_train),
                                           y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights_array))
print("✅ class_weight:", class_weight_dict)

# 5) En iyi ResNet101 modelini yükle
model_path = 'artifacts/resnet101_eniyi.h5'
model = load_model(model_path)

# 6) Son 100 katmanı eğitime aç
for layer in model.layers[:-100]:
    layer.trainable = False
for layer in model.layers[-100:]:
    layer.trainable = True
print("✅ Son 100 katman açıldı")

# 7) Compile
model.compile(
    optimizer=Adam(learning_rate=3e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 8) Callbacks
checkpoint_path = 'artifacts/resnet101_ft100_final.h5'
checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# 9) Eğitimi başlat
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    initial_epoch=START_EPOCH,
    callbacks=[checkpoint, reduce_lr],
    class_weight=class_weight_dict
)


In [ ]:
# 1) Drive ve kütüphaneler

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

# 2) Sabitler
IMG_SIZE     = 299
BATCH_SIZE   = 32
START_EPOCH  = 200
FINAL_EPOCH  = 230
BASE_DIR     = 'data/split'
TRAIN_DIR    = os.path.join(BASE_DIR, 'train')

# 3) Sadeleştirilmiş veri artırımı
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.15,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    fill_mode='nearest'
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# 4) class_weight hesapla
y_train = train_gen.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(zip(np.unique(y_train), class_weights_array))
print("✅ class_weight:", class_weights)

# 5) Modeli yükle, son 50 katmanı aç
model_path = 'artifacts/resnet101_ft100_final.h5'
model = load_model(model_path)

for layer in model.layers[:-50]:
    layer.trainable = False
for layer in model.layers[-50:]:
    layer.trainable = True

print("✅ Son 50 katman açıldı, model compile ediliyor...")

# 6) Compile et
model.compile(
    optimizer=Adam(learning_rate=1e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 7) Callbacks
checkpoint = ModelCheckpoint(
    'artifacts/resnet101_classweight_230epoch.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-8,
    verbose=1
)

# 8) Eğitim başlat
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=FINAL_EPOCH,
    initial_epoch=START_EPOCH,
    callbacks=[checkpoint, reduce_lr],
    class_weight=class_weights
)


In [ ]:
# 1) Gerekli kütüphaneler ve Drive bağlantısı

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    roc_curve, auc, f1_score, recall_score, precision_score
)
from sklearn.preprocessing import label_binarize

# 2) Modeli yükle (en iyi ağırlıklarla kaydedilen)
model_path = 'artifacts/resnet101_classweight_230epoch.h5'
model = load_model(model_path)

# 3) Test verisini yükle
IMG_SIZE = model.input_shape[1]
BATCH_SIZE = 32
TEST_DIR = 'data/split/test'

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# 4) Performans değerlendirmesi
loss, acc = model.evaluate(test_generator, verbose=1)
print(f"\n✅ Test Loss: {loss:.4f} — Test Accuracy: {acc:.4f}")

# 5) Tahminler
y_prob = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes
labels = list(test_generator.class_indices.keys())

# 6) Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Tahmin'); plt.ylabel('Gerçek')
plt.title('Confusion Matrix - ResNet101 ClassWeight')
plt.show()

# 7) Classification Report + Accuracy
print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=labels, digits=4, zero_division=0))
print(f"Overall Accuracy: {accuracy_score(y_true, y_pred):.4f}\n")

# 8) Sınıf Bazlı Metrikler
f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
recall = recall_score(y_true, y_pred, average=None, zero_division=0)
precision = precision_score(y_true, y_pred, average=None, zero_division=0)

specificity = []
for i in range(len(labels)):
    tn = np.sum((y_true != i) & (y_pred != i))
    fp = np.sum((y_true != i) & (y_pred == i))
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    specificity.append(spec)

metrics_df = pd.DataFrame({
    'Sınıf': labels,
    'F1-score': f1,
    'Sensitivity': recall,
    'Specificity': specificity,
    'Precision': precision
})
print("=== Sınıf Bazlı Metrikler ===")
print(metrics_df.round(4))

# 9) ROC Eğrileri ve AUC
y_true_bin = label_binarize(y_true, classes=np.arange(len(labels)))
plt.figure(figsize=(8,6))
for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{label} (AUC = {roc_auc:.2f})")
plt.plot([0,1], [0,1], 'k--', lw=1)
plt.title('ROC Eğrileri - ResNet101 ClassWeight')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid()
plt.tight_layout()
plt.show()
